# FinTriage AI — DistilBERT fine-tuning on Kaggle GPU

This notebook exists because fine-tuning `distilbert-base-uncased` on 99,458
narratives is not practical on the development machine: benchmarked at
**49.6 s/step** on Apple M2 MPS (batch 32, max_len 256), a 2-epoch run would take
about **86 hours**. On a Kaggle T4 the same run is well under an hour.

It runs the *same* code as the local pipeline, from the project repository, so
the split, the label order, the class weights and the hyperparameters are
identical and the four models stay comparable.

**Before running:** attach the dataset
`shashwatwork/consume-complaints-dataset-fo-nlp` and set the accelerator to
**GPU T4 x2** (or P100). Internet must be on.


## 1. Environment check

In [ ]:
import subprocess, sys, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU reported")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn on the GPU accelerator before running this notebook."


## 2. Get the project code

The pipeline lives in the repository; nothing is re-implemented here.

In [ ]:
import os, pathlib, subprocess

REPO = "https://github.com/fariskishtah/fintriage-ai"
WORK = pathlib.Path("/kaggle/working/fintriage-ai")
if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(WORK)], check=True)
os.chdir(WORK)
print(subprocess.run(["git","log","-1","--format=%h %s"], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "pyarrow==18.1.0" "PyYAML==6.0.2" 2>&1 | tail -2


## 3. Point the pipeline at the attached Kaggle dataset

The repository does not ship the raw CSV (it is not ours to redistribute), so we
copy it from the attached Kaggle dataset into `data/raw/` where the pipeline
expects it.

In [ ]:
import shutil, pathlib, pandas as pd

SRC = pathlib.Path("/kaggle/input/consume-complaints-dataset-fo-nlp/complaints_processed.csv")
assert SRC.exists(), f"attach the dataset first; {SRC} not found"
DST = pathlib.Path("data/raw/complaints_processed.csv")
DST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(SRC, DST)
df = pd.read_csv(DST)
print(f"{len(df):,} rows, columns={list(df.columns)}")
print(df["product"].value_counts())


## 4. Rebuild the splits

Deterministic: same seed, same MinHash constants, so these splits are
byte-identical to the local ones. Expect 162,421 → 124,328 rows and
99,458 / 12,435 / 12,435.

In [ ]:
!python scripts/prepare_data.py


In [ ]:
import json
q = json.load(open("results/metrics/data_quality_report.json"))
m = json.load(open("results/metrics/split_manifest.json"))
print("rows after cleaning:", f"{q['n_output']:,}")
print("exact duplicates removed:", f"{q['n_exact_duplicates']:,}")
print("near-duplicate groups:", f"{q['near_duplicate_groups']:,}",
      f"covering {q['rows_in_multi_member_groups']:,} rows")
print("split sizes:", m["sizes"])
assert q["n_output"] == 124328, "split differs from the local run - do not compare metrics"


## 5. Smoke run first

Cheap wiring check before spending GPU time on the real run.

In [ ]:
!python scripts/train_distilbert.py --smoke 512 --device cuda


## 6. The real fine-tune

2 epochs, batch 32, lr 3e-5, max_len 256, class-weighted loss, warmup 6%.
Checkpoints every 500 steps, so a session timeout resumes with `--resume`
instead of starting over.

In [ ]:
!python scripts/train_distilbert.py --device cuda --save-every 500


## 7. Also train the recurrent models here

They train in minutes and it keeps every model's timing measured on the same
hardware, which the comparison table needs to be fair.

In [ ]:
for cell in ["SimpleRNN", "LSTM", "GRU"]:
    print(f"===== {cell} =====")
    !python scripts/train_rnn.py --cell {cell} --device cuda


## 8. Select the production model and calibrate the review threshold

Validation only. The test split is still untouched at this point.

In [ ]:
!python scripts/select_model.py


## 9. Final evaluation on the held-out test set

In [ ]:
!python scripts/evaluate.py


## 10. Package the outputs

Everything needed to deploy: the fine-tuned checkpoint, the tokenizer, the RNN
weights and all the metrics. Download `fintriage_artifacts.zip` from the
notebook output, unzip into the project root, and the app will load it.

In [ ]:
import shutil, pathlib, json

out = pathlib.Path("/kaggle/working")
staging = out / "fintriage_release"
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

shutil.copytree("artifacts/DistilBERT", staging / "artifacts/DistilBERT",
                ignore=shutil.ignore_patterns("*.npy"))
for cell in ["SimpleRNN", "LSTM", "GRU"]:
    src = pathlib.Path("artifacts") / cell
    if src.exists():
        shutil.copytree(src, staging / "artifacts" / cell,
                        ignore=shutil.ignore_patterns("*.npy"))
shutil.copytree("results", staging / "results")

shutil.make_archive(str(out / "fintriage_artifacts"), "zip", staging)
size = (out / "fintriage_artifacts.zip").stat().st_size / 1e6
print(f"fintriage_artifacts.zip  {size:.0f} MB")
print(json.dumps(json.load(open("results/metrics/model_selection.json")), indent=2))


## 11. Push the model to the Hugging Face Hub (optional)

Only needed to host the checkpoint for the Space. Add a Kaggle secret named
`HF_TOKEN` with a write-scoped token; skip this cell otherwise and upload the
zip by hand.

In [ ]:
from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("no HF_TOKEN secret set, skipping:", exc)
    token = None

if token:
    !pip install -q huggingface_hub
    from huggingface_hub import HfApi
    repo_id = "CHANGE-ME/fintriage-distilbert"   # set your own namespace
    api = HfApi(token=token)
    api.create_repo(repo_id, repo_type="model", exist_ok=True, private=False)
    api.upload_folder(folder_path="artifacts/DistilBERT", repo_id=repo_id,
                      ignore_patterns=["*.npy"])
    print("pushed to", repo_id)
